# Conformal TB Triage: Extract Frozen Embeddings

**Runtime: T4 GPU** (Runtime → Change runtime type → T4 GPU)

Extracts frozen embeddings from 7 foundation models for all images in `splits.parquet`.
Each model runs in its own cell — if a session disconnects, re-run Cell 1 (setup) then skip to whichever model you need.

**Expected time:** ~5-15 min per model, ~60-90 min total.

**Output:** One parquet file per model in `conformal-tb-triage/data/interim/embeddings/`

In [ ]:
# ── Cell 1: Setup (run first every session) ───────────────────────────

import subprocess, sys, os

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
print('Installing packages...', flush=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'torchvision', 'transformers', 'timm', 'open_clip_torch',
    'torchxrayvision', 'pandas', 'pyarrow', 'Pillow', 'tqdm'],
    capture_output=True)
print('Packages installed.', flush=True)

import torch
import torchvision.transforms as T
import pandas as pd
import numpy as np
import hashlib
import gc
import time
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)', flush=True)
else:
    print('WARNING: No GPU detected. Extraction will be very slow.', flush=True)

# Paths
DRIVE_ROOT = Path('/content/drive/MyDrive/conformal-tb-triage')
EMBEDDINGS_DIR = DRIVE_ROOT / 'data' / 'interim' / 'embeddings'
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

# Load splits
print('Loading splits...', flush=True)
splits = pd.read_parquet(DRIVE_ROOT / 'data' / 'processed' / 'splits.parquet')
print(f'Loaded: {len(splits)} images across {splits["dataset"].nunique()} datasets', flush=True)
print(splits['dataset'].value_counts().to_string(), flush=True)
print(f'\nEmbeddings will be saved to: {EMBEDDINGS_DIR}')

In [ ]:
# ── Cell 1b: Copy images from Drive to local disk (FAST) ──────────────
# Uses shell cp -r which is much faster than Python per-file copy on Drive FUSE.
# Interrupt the old Cell 1b if it's still running.

import shutil

LOCAL_IMG = Path('/content/local_images')
LOCAL_IMG.mkdir(parents=True, exist_ok=True)

RAW = DRIVE_ROOT / 'data' / 'raw'

# Copy each dataset folder in one shell command (bulk copy is 10-50x faster)
dataset_dirs = [
    ('shenzhen_montgomery', RAW / 'shenzhen_montgomery'),
    ('tbx11k',              RAW / 'tbx11k'),
    ('chexpert_subset',     RAW / 'chexpert_subset'),
    ('mendeley_pakistan',    RAW / 'mendeley_pakistan'),
]

for name, src_dir in dataset_dirs:
    dst_dir = LOCAL_IMG / name
    if dst_dir.exists() and any(dst_dir.rglob('*.png')) or any(dst_dir.rglob('*.jpg')):
        n = sum(1 for _ in dst_dir.rglob('*') if _.is_file())
        print(f'{name}: already cached ({n} files). Skipping.', flush=True)
        continue

    if not src_dir.exists():
        print(f'{name}: source not found at {src_dir}. Skipping.', flush=True)
        continue

    print(f'{name}: copying from Drive...', flush=True)
    t0 = time.time()
    # Shell cp -r is significantly faster than Python shutil for Drive FUSE
    os.system(f'cp -r "{src_dir}" "{dst_dir}"')
    elapsed = time.time() - t0
    n = sum(1 for _ in dst_dir.rglob('*') if _.is_file()) if dst_dir.exists() else 0
    print(f'  Done: {n} files in {elapsed:.0f}s', flush=True)

# Remap splits file paths to local copies
def remap_to_local(file_path, dataset):
    p = Path(file_path)
    # Find the file under the local dataset directory
    local_base = LOCAL_IMG / dataset
    if not local_base.exists():
        # Try parent dataset name mapping
        if dataset in ('shenzhen', 'montgomery'):
            local_base = LOCAL_IMG / 'shenzhen_montgomery'
        elif dataset == 'chexpert':
            local_base = LOCAL_IMG / 'chexpert_subset'
        elif dataset == 'pakistan':
            local_base = LOCAL_IMG / 'mendeley_pakistan'

    if local_base.exists():
        # Try to find the same filename under local_base
        matches = list(local_base.rglob(p.name))
        if matches:
            return str(matches[0])

    return file_path  # fallback to Drive

splits['file_path'] = splits.apply(
    lambda row: remap_to_local(row['file_path'], row['dataset']), axis=1
)

n_local = splits['file_path'].str.startswith('/content/local').sum()
n_drive = len(splits) - n_local
print(f'\nPath mapping: {n_local} local, {n_drive} still on Drive', flush=True)
if n_drive > 0:
    drive_samples = splits[~splits['file_path'].str.startswith('/content/local')].head(5)
    print('Still on Drive:', flush=True)
    for _, row in drive_samples.iterrows():
        print(f'  {row["dataset"]}: {row["file_path"]}', flush=True)

In [ ]:
# ── Cell 2: Extraction helper ─────────────────────────────────────────

def batch_preprocess(preprocess, images, device):
    """
    Preprocess a list of PIL images into a batched tensor on device.
    Handles HF processors, torchvision transforms, and custom callables.
    """
    # Attempt 1: HF-style batch call with images= kwarg
    try:
        result = preprocess(images=images, return_tensors='pt')
        if hasattr(result, 'pixel_values'):
            return result['pixel_values'].to(device)
    except (TypeError, AttributeError):
        pass

    # Attempt 2: HF-style positional arg
    try:
        result = preprocess(images, return_tensors='pt')
        if hasattr(result, 'pixel_values'):
            return result['pixel_values'].to(device)
    except (TypeError, AttributeError):
        pass

    # Attempt 3: Per-image, handling both tensor and BatchFeature returns
    tensors = []
    for img in images:
        out = preprocess(img)
        if isinstance(out, torch.Tensor):
            tensors.append(out)
        elif hasattr(out, 'pixel_values'):
            tensors.append(out['pixel_values'].squeeze(0))
        elif hasattr(out, 'data') and 'pixel_values' in out.data:
            tensors.append(out.data['pixel_values'].squeeze(0))
        else:
            raise ValueError(f'Preprocessor returned unexpected type: {type(out)}')
    return torch.stack(tensors).to(device)


def smoke_test(preprocess, extract_fn, model, test_image_path):
    """Run one image through the full pipeline to catch errors early."""
    print('  Smoke test (1 image)...', flush=True)
    img = Image.open(test_image_path).convert('RGB')
    batch = batch_preprocess(preprocess, [img], device)
    print(f'    Preprocessed shape: {batch.shape}, dtype: {batch.dtype}', flush=True)
    with torch.no_grad():
        emb = extract_fn(model, batch)
    print(f'    Embedding shape: {emb.shape} -> {emb.shape[1]}d', flush=True)
    print('  Smoke test passed.', flush=True)
    return emb.shape[1]


def extract_embeddings(
    model,
    preprocess,
    extract_fn,
    model_name,
    splits_df,
    output_dir,
    batch_size=64,
):
    output_path = output_dir / f'{model_name}.parquet'

    # Skip if already done
    if output_path.exists():
        existing = pd.read_parquet(output_path)
        if len(existing) >= len(splits_df) * 0.95:
            print(f'{model_name}: Already extracted ({len(existing)} rows). Skipping.', flush=True)
            return output_path
        else:
            print(f'{model_name}: Partial ({len(existing)}/{len(splits_df)}). Re-extracting.', flush=True)

    model.eval()
    all_embeddings = []
    all_ids = []
    failed = []

    file_paths = splits_df['file_path'].tolist()
    patient_ids = splits_df['patient_id'].tolist()

    print(f'{model_name}: Extracting {len(file_paths)} images (batch_size={batch_size})...', flush=True)
    t0 = time.time()
    pbar = tqdm(total=len(file_paths), desc=model_name, unit='img', mininterval=2)

    for start in range(0, len(file_paths), batch_size):
        batch_paths = file_paths[start:start + batch_size]
        batch_ids = patient_ids[start:start + batch_size]

        # Load images
        images = []
        valid_ids = []
        for fpath, pid in zip(batch_paths, batch_ids):
            try:
                img = Image.open(fpath).convert('RGB')
                images.append(img)
                valid_ids.append(pid)
            except Exception as e:
                failed.append((fpath, str(e)))

        if not images:
            pbar.update(len(batch_paths))
            continue

        batch_tensor = batch_preprocess(preprocess, images, device)

        with torch.no_grad():
            emb = extract_fn(model, batch_tensor)

        all_embeddings.append(emb.cpu().float().numpy())
        all_ids.extend(valid_ids)
        pbar.update(len(batch_paths))

        elapsed_so_far = time.time() - t0
        imgs_done = len(all_ids)
        speed = imgs_done / elapsed_so_far if elapsed_so_far > 0 else 0
        pbar.set_postfix({'img/s': f'{speed:.0f}', 'fail': len(failed)}, refresh=False)

    pbar.close()
    elapsed = time.time() - t0
    emb_matrix = np.concatenate(all_embeddings, axis=0)
    embed_dim = emb_matrix.shape[1]

    print(f'  Done: {emb_matrix.shape[0]} x {embed_dim}d in {elapsed:.1f}s ({emb_matrix.shape[0]/elapsed:.0f} img/s)', flush=True)
    if failed:
        print(f'  Failed: {len(failed)} images', flush=True)
        for fp, err in failed[:3]:
            print(f'    {Path(fp).name}: {err[:80]}', flush=True)

    emb_cols = [f'emb_{i}' for i in range(embed_dim)]
    emb_df = pd.DataFrame(emb_matrix, columns=emb_cols)
    emb_df.insert(0, 'patient_id', all_ids)

    meta = splits_df[['patient_id', 'split', 'dataset', 'tb_binary']].drop_duplicates('patient_id')
    emb_df = emb_df.merge(meta, on='patient_id', how='left')

    print(f'  Saving to Drive...', flush=True)
    emb_df.to_parquet(output_path, index=False)
    size_mb = output_path.stat().st_size / 1e6
    print(f'  Saved: {output_path.name} ({size_mb:.1f} MB, {len(emb_df)} rows)', flush=True)

    norms = np.linalg.norm(emb_matrix, axis=1)
    print(f'  L2 norm: mean={norms.mean():.2f}, std={norms.std():.2f}, range=[{norms.min():.2f}, {norms.max():.2f}]', flush=True)

    return output_path


def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        mem_used = torch.cuda.memory_allocated() / 1e9
        mem_total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'  GPU memory: {mem_used:.2f} / {mem_total:.1f} GB', flush=True)

print('Extraction helpers defined.')

In [ ]:
# ── Cell 3: RAD-DINO (primary model) ──────────────────────────────────
# ViT-B/16, 768d, microsoft/rad-dino

from transformers import AutoModel, AutoImageProcessor

print('Downloading RAD-DINO from HuggingFace (first run only)...', flush=True)
t0 = time.time()
processor = AutoImageProcessor.from_pretrained('microsoft/rad-dino', use_fast=False)
model = AutoModel.from_pretrained('microsoft/rad-dino').to(device)
model.eval()
print(f'  Loaded in {time.time()-t0:.1f}s. Parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M', flush=True)

def rad_dino_extract(model, batch):
    outputs = model(pixel_values=batch)
    return outputs.last_hidden_state[:, 0, :]

# Smoke test before full extraction
smoke_test(processor, rad_dino_extract, model, splits.iloc[0]['file_path'])

extract_embeddings(
    model=model,
    preprocess=processor,
    extract_fn=rad_dino_extract,
    model_name='rad_dino',
    splits_df=splits,
    output_dir=EMBEDDINGS_DIR,
    batch_size=64,
)

del model, processor
free_gpu()

In [ ]:
# ── Cell 4: EVA-X-B ───────────────────────────────────────────────────
# ViT-B/16, 768d. Weights are raw .pt file on HuggingFace — need manual load.

import timm
from huggingface_hub import hf_hub_download

print('Downloading EVA-X-B weights...', flush=True)
t0 = time.time()

ckpt_path = hf_hub_download(
    'MapleF/eva_x',
    filename='eva_x_base_patch16_merged520k_mim.pt'
)
print(f'  Downloaded checkpoint: {Path(ckpt_path).stat().st_size/1e6:.1f} MB', flush=True)

model = timm.create_model('vit_base_patch16_224', pretrained=False, num_classes=0)

# weights_only=False needed because checkpoint contains numpy objects
state_dict = torch.load(ckpt_path, map_location='cpu', weights_only=False)
if 'model' in state_dict:
    state_dict = state_dict['model']
elif 'state_dict' in state_dict:
    state_dict = state_dict['state_dict']

result = model.load_state_dict(state_dict, strict=False)
print(f'  Loaded weights. Missing: {len(result.missing_keys)}, Unexpected: {len(result.unexpected_keys)}', flush=True)
if result.missing_keys:
    print(f'    Missing (first 5): {result.missing_keys[:5]}', flush=True)
if result.unexpected_keys:
    print(f'    Unexpected (first 5): {result.unexpected_keys[:5]}', flush=True)

model = model.to(device)
model.eval()
print(f'  Ready in {time.time()-t0:.1f}s. Parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M', flush=True)

eva_transform = T.Compose([
    T.Resize(224, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def eva_extract(model, batch):
    features = model.forward_features(batch)
    if features.dim() == 3:
        return features[:, 0, :]
    return features

smoke_test(eva_transform, eva_extract, model, splits.iloc[0]['file_path'])

extract_embeddings(
    model=model,
    preprocess=eva_transform,
    extract_fn=eva_extract,
    model_name='eva_x',
    splits_df=splits,
    output_dir=EMBEDDINGS_DIR,
    batch_size=64,
)

del model
free_gpu()

In [ ]:
# ── Cell 5: CheXzero ──────────────────────────────────────────────────
# ViT-B/32 (CLIP vision encoder), 512d
# CheXzero is a CLIP model fine-tuned on MIMIC-CXR. The weights are
# distributed via the original GitHub repo, not as a standard HF model.

import open_clip
from huggingface_hub import hf_hub_download

print('Loading CheXzero...', flush=True)
t0 = time.time()

# CheXzero is built on OpenAI CLIP ViT-B/32 with fine-tuned weights
model, _, preprocess_clip = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')

# Try to download CheXzero fine-tuned weights
chexzero_loaded = False
# The original repo distributes weights via Google Drive / GitHub releases
# Try common HuggingFace locations
for weight_file in ['pytorch_model.bin', 'model.pt', 'chexzero_weights.pt', 'best_model.pt']:
    try:
        ckpt_path = hf_hub_download('rajpurkarlab/CheXzero', filename=weight_file)
        state = torch.load(ckpt_path, map_location='cpu', weights_only=False)
        if 'model_state_dict' in state:
            state = state['model_state_dict']
        elif 'state_dict' in state:
            state = state['state_dict']
        model.load_state_dict(state, strict=False)
        chexzero_loaded = True
        print(f'  Loaded CheXzero weights from {weight_file}', flush=True)
        break
    except Exception:
        continue

if not chexzero_loaded:
    print('  Could not find CheXzero fine-tuned weights on HuggingFace.', flush=True)
    print('  Using OpenAI CLIP ViT-B/32 base weights as fallback.', flush=True)
    print('  NOTE: This is NOT the CheXzero model. Results labelled accordingly.', flush=True)

model = model.to(device)
model.eval()
print(f'  Loaded in {time.time()-t0:.1f}s. Parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M', flush=True)

def chexzero_extract(model, batch):
    return model.encode_image(batch)

smoke_test(preprocess_clip, chexzero_extract, model, splits.iloc[0]['file_path'])

extract_embeddings(
    model=model,
    preprocess=preprocess_clip,
    extract_fn=chexzero_extract,
    model_name='chexzero',
    splits_df=splits,
    output_dir=EMBEDDINGS_DIR,
    batch_size=64,
)

del model
free_gpu()

In [ ]:
# ── Cell 6: BiomedCLIP ────────────────────────────────────────────────
# ViT-B/16, 512d, microsoft/BiomedCLIP

import open_clip

print('Downloading BiomedCLIP (first run only)...', flush=True)
t0 = time.time()
model, _, preprocess_biomed = open_clip.create_model_and_transforms(
    'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
)
model = model.to(device)
model.eval()
print(f'  Loaded in {time.time()-t0:.1f}s. Parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M', flush=True)

def biomedclip_extract(model, batch):
    return model.encode_image(batch)

smoke_test(preprocess_biomed, biomedclip_extract, model, splits.iloc[0]['file_path'])

extract_embeddings(
    model=model,
    preprocess=preprocess_biomed,
    extract_fn=biomedclip_extract,
    model_name='biomedclip',
    splits_df=splits,
    output_dir=EMBEDDINGS_DIR,
    batch_size=64,
)

del model
free_gpu()

In [ ]:
# ── Cell 7: GLoRIA ────────────────────────────────────────────────────
# ResNet-50 image encoder

print('Loading GLoRIA image encoder...', flush=True)
t0 = time.time()

gloria_loaded = False

# Attempt 1: gloria pip package
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gloria'], capture_output=True)
    import gloria as gloria_pkg
    model = gloria_pkg.load_img_model()
    model = model.to(device)
    model.eval()
    gloria_loaded = True
    print(f'  Loaded via gloria package in {time.time()-t0:.1f}s.', flush=True)
except Exception as e1:
    print(f'  gloria package failed: {e1}', flush=True)

# Attempt 2: HuggingFace
if not gloria_loaded:
    try:
        from transformers import AutoModel
        model = AutoModel.from_pretrained('marshuang80/gloria').to(device)
        model.eval()
        gloria_loaded = True
        print(f'  Loaded via HuggingFace in {time.time()-t0:.1f}s.', flush=True)
    except Exception as e2:
        print(f'  HuggingFace failed: {e2}', flush=True)

# Attempt 3: ImageNet ResNet-50 fallback
if not gloria_loaded:
    import torchvision.models as models
    print('  Using ImageNet ResNet-50 as fallback (NOT GLoRIA weights).', flush=True)
    resnet = models.resnet50(weights='IMAGENET1K_V1')
    model = torch.nn.Sequential(*list(resnet.children())[:-1], torch.nn.Flatten())
    model = model.to(device)
    model.eval()

print(f'  Parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M', flush=True)

gloria_transform = T.Compose([
    T.Resize(224, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def gloria_extract(model, batch):
    out = model(batch)
    if hasattr(out, 'pooler_output') and out.pooler_output is not None:
        return out.pooler_output
    if hasattr(out, 'last_hidden_state'):
        return out.last_hidden_state[:, 0, :]
    if isinstance(out, tuple):
        out = out[0]
    if out.dim() > 2:
        out = out.squeeze(-1).squeeze(-1)
    return out

smoke_test(gloria_transform, gloria_extract, model, splits.iloc[0]['file_path'])

extract_embeddings(
    model=model,
    preprocess=gloria_transform,
    extract_fn=gloria_extract,
    model_name='gloria',
    splits_df=splits,
    output_dir=EMBEDDINGS_DIR,
    batch_size=64,
)

del model
free_gpu()

In [ ]:
# ── Cell 8: torchxrayvision (supervised baseline) ─────────────────────
# DenseNet-121, 1024d features

import torchxrayvision as xrv

print('Loading torchxrayvision DenseNet-121 ("all" weights)...', flush=True)
t0 = time.time()
model = xrv.models.DenseNet(weights='densenet121-res224-all')
model = model.to(device)
model.eval()
print(f'  Loaded in {time.time()-t0:.1f}s. Parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M', flush=True)

def xrv_preprocess(img):
    img_np = np.array(img.convert('L'), dtype=np.float32)
    from PIL import Image as PILImage
    img_resized = PILImage.fromarray(img_np).resize((224, 224))
    img_np = np.array(img_resized, dtype=np.float32)
    img_np = (img_np / 255.0 * 2048) - 1024
    tensor = torch.from_numpy(img_np).unsqueeze(0)  # [1, H, W]
    return tensor

def xrv_extract(model, batch):
    # model.features() returns [B, 1024, 7, 7] feature maps
    # Apply global average pooling to get [B, 1024]
    feat_maps = model.features(batch)
    pooled = feat_maps.mean(dim=[2, 3])  # global average pool over spatial dims
    return pooled

smoke_test(xrv_preprocess, xrv_extract, model, splits.iloc[0]['file_path'])

extract_embeddings(
    model=model,
    preprocess=xrv_preprocess,
    extract_fn=xrv_extract,
    model_name='torchxrayvision',
    splits_df=splits,
    output_dir=EMBEDDINGS_DIR,
    batch_size=64,
)

del model
free_gpu()

In [ ]:
# ── Cell 9: DINOv2-B (non-medical control) ────────────────────────────
# ViT-B/14, 768d, facebook/dinov2-base

from transformers import AutoModel, AutoImageProcessor

print('Downloading DINOv2-B (first run only)...', flush=True)
t0 = time.time()
processor = AutoImageProcessor.from_pretrained('facebook/dinov2-base', use_fast=False)
model = AutoModel.from_pretrained('facebook/dinov2-base').to(device)
model.eval()
print(f'  Loaded in {time.time()-t0:.1f}s. Parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M', flush=True)

def dinov2_extract(model, batch):
    outputs = model(pixel_values=batch)
    return outputs.last_hidden_state[:, 0, :]

smoke_test(processor, dinov2_extract, model, splits.iloc[0]['file_path'])

extract_embeddings(
    model=model,
    preprocess=processor,
    extract_fn=dinov2_extract,
    model_name='dinov2',
    splits_df=splits,
    output_dir=EMBEDDINGS_DIR,
    batch_size=64,
)

del model, processor
free_gpu()

In [ ]:
# ── Cell 10: Verification ─────────────────────────────────────────────

print('Embedding extraction verification')
print('=' * 70)

expected_models = ['rad_dino', 'eva_x', 'chexzero', 'biomedclip', 'gloria', 'torchxrayvision', 'dinov2']

for name in expected_models:
    path = EMBEDDINGS_DIR / f'{name}.parquet'
    if path.exists():
        df = pd.read_parquet(path)
        emb_cols = [c for c in df.columns if c.startswith('emb_')]
        size_mb = path.stat().st_size / 1e6
        print(f'  {name:20s}  {len(df):6d} images  {len(emb_cols):4d}d  {size_mb:6.1f} MB  OK')
    else:
        print(f'  {name:20s}  MISSING')

print()
total_size = sum((EMBEDDINGS_DIR / f'{n}.parquet').stat().st_size
                 for n in expected_models
                 if (EMBEDDINGS_DIR / f'{n}.parquet').exists()) / 1e6
print(f'Total embeddings size: {total_size:.1f} MB')
print(f'Location: {EMBEDDINGS_DIR}')
print()
print('Next: probe training + conformal calibration (CPU, no GPU needed).')

In [ ]:
# ── Cell 11: FP16 consistency check (pre-registered) ──────────────────
# Run AFTER all models are extracted. Uses RAD-DINO as the test case.

from transformers import AutoModel, AutoImageProcessor

print('FP16 vs FP32 consistency check (RAD-DINO, 200 images)...', flush=True)

processor = AutoImageProcessor.from_pretrained('microsoft/rad-dino', use_fast=False)
model = AutoModel.from_pretrained('microsoft/rad-dino').to(device)
model.eval()

fp32_df = pd.read_parquet(EMBEDDINGS_DIR / 'rad_dino.parquet')
emb_cols = [c for c in fp32_df.columns if c.startswith('emb_')]

sample = splits.head(200)
fp16_embeddings = []

for _, row in tqdm(sample.iterrows(), total=len(sample), desc='FP16 extraction'):
    try:
        img = Image.open(row['file_path']).convert('RGB')
        inputs = processor(images=img, return_tensors='pt').to(device)
        with torch.no_grad():
            with torch.amp.autocast('cuda'):
                outputs = model(**inputs)
                emb = outputs.last_hidden_state[:, 0, :].cpu().float().numpy()
        fp16_embeddings.append(emb.squeeze())
    except:
        fp16_embeddings.append(np.zeros(len(emb_cols)))

fp16_matrix = np.stack(fp16_embeddings)

fp32_sample = fp32_df[fp32_df['patient_id'].isin(sample['patient_id'])]
fp32_matrix = fp32_sample[emb_cols].values[:len(fp16_matrix)]

max_diff = np.max(np.abs(fp32_matrix - fp16_matrix))
mean_diff = np.mean(np.abs(fp32_matrix - fp16_matrix))
cosine_sims = np.array([
    np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)
    for a, b in zip(fp32_matrix, fp16_matrix)
])

print(f'\nFP16 vs FP32 comparison (RAD-DINO, n={len(fp16_matrix)}):')
print(f'  Max absolute difference per dimension: {max_diff:.6f}')
print(f'  Mean absolute difference per dimension: {mean_diff:.6f}')
print(f'  Cosine similarity: mean={cosine_sims.mean():.6f}, min={cosine_sims.min():.6f}')

if max_diff < 1e-3:
    print(f'  PASS: Max diff {max_diff:.6f} < 1e-3 threshold. FP16 is reliable.')
else:
    print(f'  FAIL: Max diff {max_diff:.6f} >= 1e-3. FP32 mandatory for this model.')

del model, processor
free_gpu()